# 02 · Validate

Before asking *why do the models disagree?*, ask *can I trust what I'm comparing?* Three families of gates: **data**, **model-output** and **reconciliation**. A failing gate doesn't stop the run — it is surfaced, so no one silently trusts something broken.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from src.synthetic import generate
from src import validation as V
members, _ = generate()
data_gates = V.gates_to_frame(V.run_data_gates(members))
data_gates[['gate','status','value','limit','detail']]

,gate,status,value,limit,detail
0,unique_member_ids,PASS,0.0000,== 0,0 duplicated member_id rows
1,expected_schema,PASS,0.0000,== 0,all required columns present
2,acceptable_missingness,PASS,0.0000,<= 0.01,rows with any null in required fields (last_cl...
3,claims_completeness,WARN,0.0664,<= 0.05,share of members with claims_completeness < 0.8
4,date_freshness,PASS,0.0000,<= 0.0,dates after the reference snapshot 2025-12-31
5,duplicate_detection,FAIL,0.0288,<= 0.005,members with >= 8% duplicate claim lines
6,score_ranges,PASS,0.0000,== 0,"scores outside [0.0, 15.0] or null"


Two gates fire on this dataset, exactly where defects were planted: **duplicate detection** (FAIL) and **claims completeness** (WARN). Identifier, schema, freshness and range checks are clean.

In [2]:
# How the gates behave on a broken table (what they are for)
broken = pd.concat([members, members.iloc[:5]])
broken = broken.drop(columns=['claims_completeness'])
print(V.check_unique_ids(broken).status, '|', V.check_schema(broken).status)

FAIL | FAIL


In [3]:
# Population Stability Index sanity check: identical -> 0, shifted -> large
x = np.random.default_rng(0).normal(2, 1, 5000)
print(round(V.psi(x, x), 4), round(V.psi(x, x + 1.5), 3))

0.0 1.994
